In [1]:
# Pyomo is an algebraic modeling language for Python. It lets us describe
# optimization problems (sets, parameters, variables, constraints, objective)
# in a solver-agnostic way, then hand the assembled model to any compatible
# solver (HiGHS, CBC, Gurobi, CPLEX, ...).
import pyomo.environ as pyo

Sets

$\mathcal{I} = \{$items$\}$

Parameters

$v_i$ value of item $i \in \mathcal{I}$ \
$w_i$ weight of item $i \in \mathcal{I}$ \
$W$ knapsack weight capacity

Variables

$y_i \in \{0, 1\}$ whether item $i$ is packed

Objective and Constraints

\begin{gather}
 \max_y \sum_{i \in \mathcal{I}} v_i y_i \;\;\;\textrm{(value)}\\
 \textrm{s.t.} \,\, \sum_{i \in \mathcal{I}} w_i y_i \le W \;\;\textrm{(capacity)} \\
 y_i \in \{0, 1\} \;\; \forall i \in \mathcal{I}
\end{gather}

In [2]:
# Item catalog: each entry has a value (utility we get from packing it)
# and a weight (capacity it consumes in the knapsack).
data = {
    'laptop':        {'value': 25, 'weight':  6},
    'water_bottle':  {'value':  4, 'weight':  2},
    'tent':          {'value': 50, 'weight': 11},
    'sleeping_bag':  {'value': 14, 'weight':  5},
    'flashlight':    {'value':  6, 'weight':  1},
    'first_aid_kit': {'value': 10, 'weight':  3},
    'stove':         {'value': 12, 'weight':  4},
    'jacket':        {'value': 11, 'weight':  4},
    'map':           {'value':  5, 'weight':  1},
    'camera':        {'value': 16, 'weight':  3},
    'knife':         {'value':  8, 'weight':  1},
    'compass':       {'value':  7, 'weight':  1},
}

# Total weight capacity of the knapsack.
weight_limit = 23

In [3]:
# Build a concrete Pyomo model (all data known up front, no abstract symbols).
m = pyo.ConcreteModel()

# Index set: the names of the candidate items.
m.things = pyo.Set(initialize=data.keys())

# Decision variable: y[i] = 1 if item i is packed, 0 otherwise
# (binary => 0/1 only).
m.y = pyo.Var(m.things, within=pyo.Binary)

# Objective: maximize the total value of the items selected.
m.value = pyo.Objective(
    expr=sum(data[i]['value'] * m.y[i] for i in m.things),
    sense=pyo.maximize,
)

# Capacity constraint: total weight of chosen items must not exceed the limit.
m.weight = pyo.Constraint(
    expr=sum(data[i]['weight'] * m.y[i] for i in m.things) <= weight_limit
)

In [4]:
# Use HiGHS as the MILP solver (ships as a pip wheel via highspy).
solver = pyo.SolverFactory('appsi_highs')

# Solve the model; tee=True streams the solver's log to stdout.
results = solver.solve(m, tee=True)

print("\nOptimal things:")

# Print every item the solver picked. The 0.5 threshold guards against
# tiny floating-point noise around an integer 0/1 value.
for i in m.things:
    if pyo.value(m.y[i]) > 0.5:
        print(i)

# Optimal objective value (sum of values of chosen items).
print("\nTotal value =", pyo.value(m.value))

# Sanity check: total weight of the chosen items should be <= weight_limit.
print(
    "Total weight =",
    sum(data[i]['weight'] * pyo.value(m.y[i]) for i in m.things),
)

Running HiGHS 1.14.0 (git hash: 7df0786): Copyright (c) 2026 under MIT licence terms
MIP has 1 row; 12 cols; 12 nonzeros; 12 integer variables (12 binary)
Coefficient ranges:
  Matrix  [1e+00, 1e+01]
  Cost    [4e+00, 5e+01]
  Bound   [1e+00, 1e+00]
  RHS     [2e+01, 2e+01]
Presolving model
1 rows, 12 cols, 12 nonzeros 0s
1 rows, 12 cols, 12 nonzeros 0s
Presolve reductions: rows 1(-0); columns 12(-0); nonzeros 12(-0) - Not reduced
Objective function is integral with scale 1

Solving MIP model with:
   1 row
   12 cols (12 binary, 0 integer, 0 implied int., 0 continuous, 0 domain fixed)
   12 nonzeros

Src: B => Branching; C => Central rounding; F => Feasibility pump; H => Heuristic;
     I => Shifting; J => Feasibility jump; L => Sub-MIP; P => Empty MIP; R => Randomized rounding;
     S => Solve LP; T => Evaluate node; U => Unbounded; X => User solution; Y => HiGHS solution;
     Z => ZI Round; l => Trivial lower; p => Trivial point; u => Trivial upper; z => Trivial zero

        Nodes